In [ ]:
import pandas as pd
import numpy as np
import torch
import transformers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModel, BertTokenizerFast, DistilBertTokenizerFast, AutoTokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read Data

In [ ]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
df = pd.read_csv('../data/combined_letters_degendered.csv')

In [ ]:
df

,full_text,label
0,it is my pleasure to write a letter of recomme...,0
1,i am pleased to highly recommend identifier fo...,0
2,i am writing this letter in support of identif...,0
3,identifier identifier recently completed an an...,0
4,it is my pleasure to recommend dr. identifier ...,0
...,...,...
8982,it is with pleasure that i recommend first_nam...,0
8983,we are very pleased to write this letter based...,0
8984,i am writing this letter of recommendation for...,1
8985,it is my pleasure to write first_name support ...,1


# Create Training and Test Sets

In [ ]:
train_text, temp_text, train_labels, temp_labels = train_test_split(df['full_text'], df['label'],
                                                                    random_state=0,
                                                                    test_size=0.3,
                                                                    stratify=df['label'])


val_text, test_text, val_labels, test_labels = train_test_split(temp_text, temp_labels,
                                                                random_state=0,
                                                                test_size=0.5,
                                                                stratify=temp_labels)

In [ ]:
bert = AutoModel.from_pretrained('Charangan/MedBERT')
tokenizer = AutoTokenizer.from_pretrained('Charangan/MedBERT')

In [ ]:
tokens_train = tokenizer.batch_encode_plus(
    train_text.tolist(),
    padding=True,
    max_length=256,
    truncation=True
)

tokens_val = tokenizer.batch_encode_plus(
    val_text.tolist(),
    padding=True,
    max_length=256,
    truncation=True
)

tokens_test = tokenizer.batch_encode_plus(
    test_text.tolist(),
    padding=True,
    max_length=256,
    truncation=True
)

In [ ]:
train_seq = torch.tensor(tokens_train['input_ids'])
train_mask = torch.tensor(tokens_train['attention_mask'])
train_y = torch.tensor(train_labels.tolist())

val_seq = torch.tensor(tokens_val['input_ids'])
val_mask = torch.tensor(tokens_val['attention_mask'])
val_y = torch.tensor(val_labels.tolist())

test_seq = torch.tensor(tokens_test['input_ids'])
test_mask = torch.tensor(tokens_test['attention_mask'])
test_y = torch.tensor(test_labels.tolist())

In [ ]:
batch_size = 16
train_data = TensorDataset(train_seq, train_mask, train_y)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

val_data = TensorDataset(val_seq, val_mask, val_y)
val_sampler = SequentialSampler(val_data)
val_dataloader = DataLoader(val_data, sampler = val_sampler, batch_size=batch_size)

In [ ]:
# freeze weights, uncomment if you would like to unfreeze weights
for param in bert.parameters():
    param.requires_grad = False


# Create Model

In [ ]:
class BERT_Arch(nn.Module):

    def __init__(self, bert):

        super(BERT_Arch, self).__init__()

        self.bert = bert

        # dropout layer
        self.dropout = nn.Dropout(0.1)

        # relu activation function
        self.relu =  nn.ReLU()

        # dense layer 1
        self.fc1 = nn.Linear(bert.config.hidden_size,128)

        # dense layer 2 (Output layer)
        self.fc2 = nn.Linear(128,2)

        #softmax activation function
        self.softmax = nn.LogSoftmax(dim=1)

    #define the forward pass
    def forward(self, sent_id, mask):

        #pass the inputs to the model
        distilbert_output = self.bert(sent_id, attention_mask=mask)

        hidden_state = distilbert_output[0]  # (bs, seq_len, dim)
        pooled_output = hidden_state[:, 0]

        x = self.fc1(pooled_output)

        x = self.relu(x)

        x = self.dropout(x)

        # output layer
        x = self.fc2(x)

        # apply softmax activation
        x = self.softmax(x)

        return x

In [ ]:
model = BERT_Arch(bert)
model = model.to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
weights= torch.tensor(class_weights,dtype=torch.float)
weights = weights.to(device)
cross_entropy  = nn.NLLLoss(weight=weights)
epochs = 10

In [ ]:
# function to train the model
def train():

  model.train()

  total_loss, total_accuracy = 0, 0

  # empty list to save model predictions
  total_preds=[]
  total_labels=[]

  # iterate over batches
  for step,batch in enumerate(tqdm(train_dataloader, desc="Training", leave=True)):

    # # progress update after every 50 batches.
    # if step % 50 == 0 and not step == 0:
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(train_dataloader)))

    # push the batch to gpu
    batch = [r.to(device) for r in batch]

    sent_id, mask, labels = batch

    # clear previously calculated gradients
    model.zero_grad()

    # get model predictions for the current batch
    preds = model(sent_id, mask)

    # compute the loss between actual and predicted values
    loss = cross_entropy(preds, labels)

    # add on to the total loss
    total_loss = total_loss + loss.item()

    # backward pass to calculate the gradients
    loss.backward()

    # clip the the gradients to 1.0. It helps in preventing the exploding gradient problem
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    # update parameters
    optimizer.step()

    # model predictions are stored on GPU. So, push it to CPU
    preds=preds.detach().cpu().numpy()

    # append the model predictions
    total_preds.append(preds)

    labels=labels.detach().cpu().numpy()

    # append labels
    total_labels.append(labels)

  # compute the training loss of the epoch
  avg_loss = total_loss / len(train_dataloader)

  # predictions are in the form of (no. of batches, size of batch, no. of classes).
  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Training Classification Report: \n', classification_report(total_labels, epoch_preds))
  print('Training Confusion Matrix: \n', confusion_matrix(total_labels, epoch_preds))

  #returns the loss and predictions
  return avg_loss, total_preds

In [ ]:
# function for evaluating the model
def evaluate():

  print("\nEvaluating...")

  # deactivate dropout layers
  model.eval()

  total_loss, total_accuracy = 0, 0

  # empty list to save the model predictions
  total_preds = []
  total_labels = []

  # iterate over batches
  for step,batch in enumerate(val_dataloader):

    # # Progress update every 50 batches.
    # if step % 50 == 0 and not step == 0:

    #   # Calculate elapsed time in minutes.
    #   elapsed = format_time(time.time() - t0)

    #   # Report progress.
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(val_dataloader)))

    # push the batch to gpu
    batch = [t.to(device) for t in batch]

    sent_id, mask, labels = batch

    # deactivate autograd
    with torch.no_grad():

      # model predictions
      preds = model(sent_id, mask)

      # compute the validation loss between actual and predicted values
      loss = cross_entropy(preds,labels)

      total_loss = total_loss + loss.item()

      preds = preds.detach().cpu().numpy()

      total_preds.append(preds)

      labels = labels.detach().cpu().numpy()

      total_labels.append(labels)

  # compute the validation loss of the epoch
  avg_loss = total_loss / len(val_dataloader)

  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Validation Classification Report: \n', classification_report(total_labels, epoch_preds))
  print('Validation Confusion Matrix: \n', confusion_matrix(total_labels, epoch_preds))

  return avg_loss, total_preds

In [ ]:
# set initial loss to infinite
best_valid_loss = float('inf')

# empty lists to store training and validation loss of each epoch
train_losses=[]
valid_losses=[]

#for each epoch
for epoch in tqdm(range(epochs), desc="Training"):

    print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))

    #train model
    train_loss, _ = train()

    #evaluate model
    valid_loss, _ = evaluate()

    #save the best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        print('Model Saved!')
        torch.save(model, '../saved_models/saved_model.pt')

    # append training and validation loss
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f'\nTraining Loss: {train_loss:.3f}')
    print(f'Validation Loss: {valid_loss:.3f}')

Training:   0%|          | 0/10 [00:00<?, ?it/s]


 Epoch 1 / 10


Training:   0%|          | 0/394 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.32      0.49      0.39      1950
           1       0.70      0.54      0.61      4340

    accuracy                           0.52      6290
   macro avg       0.51      0.51      0.50      6290
weighted avg       0.58      0.52      0.54      6290

Training Confusion Matrix: 
 [[ 947 1003]
 [2018 2322]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.36      0.09      0.14       418
           1       0.69      0.93      0.79       930

    accuracy                           0.67      1348
   macro avg       0.52      0.51      0.47      1348
weighted avg       0.59      0.67      0.59      1348

Validation Confusion Matrix: 
 [[ 37 381]
 [ 67 863]]
Model Saved!

Training Loss: 0.696
Validation Loss: 0.697

 Epoch 2 / 10


Training:   0%|          | 0/394 [00:00<?, ?it/s]

...

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.36      0.50      0.42      1950
           1       0.73      0.59      0.65      4340

    accuracy                           0.56      6290
   macro avg       0.54      0.55      0.53      6290
weighted avg       0.61      0.56      0.58      6290

Training Confusion Matrix: 
 [[ 973  977]
 [1764 2576]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.32      0.59      0.41       418
           1       0.70      0.44      0.54       930

    accuracy                           0.49      1348
   macro avg       0.51      0.51      0.48      1348
weighted avg       0.58      0.49      0.50      1348

Validation Confusion Matrix: 
 [[245 173]
 [519 411]]

Training Loss: 0.681
Validation Loss: 0.696

 Epoch 9 / 10


Training:   0%|          | 0/394 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.37      0.55      0.44      1950
           1       0.74      0.58      0.65      4340

    accuracy                           0.57      6290
   macro avg       0.56      0.56      0.55      6290
weighted avg       0.63      0.57      0.58      6290

Training Confusion Matrix: 
 [[1076  874]
 [1836 2504]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.33      0.38      0.36       418
           1       0.70      0.66      0.68       930

    accuracy                           0.57      1348
   macro avg       0.52      0.52      0.52      1348
weighted avg       0.59      0.57      0.58      1348

Validation Confusion Matrix: 
 [[160 258]
 [319 611]]

Training Loss: 0.680
Validation Loss: 0.694

 Epoch 10 / 10


Training:   0%|          | 0/394 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.37      0.50      0.43      1950
           1       0.73      0.61      0.67      4340

    accuracy                           0.58      6290
   macro avg       0.55      0.56      0.55      6290
weighted avg       0.62      0.58      0.59      6290

Training Confusion Matrix: 
 [[ 982  968]
 [1675 2665]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.32      0.56      0.41       418
           1       0.71      0.48      0.57       930

    accuracy                           0.50      1348
   macro avg       0.52      0.52      0.49      1348
weighted avg       0.59      0.50      0.52      1348

Validation Confusion Matrix: 
 [[233 185]
 [484 446]]

Training Loss: 0.682
Validation Loss: 0.695


# Test Model

In [ ]:
model = torch.load('../saved_models/saved_model.pt', weights_only=False)

In [ ]:
model.eval()  # Set model to eval mode

# Create DataLoader for test set
test_data = TensorDataset(test_seq, test_mask, test_y)
test_dataloader = DataLoader(test_data, batch_size=32)  # adjust batch size as needed

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_dataloader:
        sent_id, mask, labels = [b.to(device) for b in batch]

        # Forward pass
        outputs = model(sent_id, mask)  # shape: (batch_size, num_classes)

        # Get predicted class (as indices)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [ ]:
  print('Test Classification Report: \n', classification_report(all_labels, all_preds))
  print('Test Confusion Matrix: \n', confusion_matrix(all_labels, all_preds))

Test Classification Report: 
               precision    recall  f1-score   support

           0       0.31      0.47      0.37       418
           1       0.69      0.53      0.60       931

    accuracy                           0.51      1349
   macro avg       0.50      0.50      0.49      1349
weighted avg       0.57      0.51      0.53      1349

Test Confusion Matrix: 
 [[196 222]
 [439 492]]
